# Neurons across environments (bat 11719)

Focused version of `preprocess-data-v0j-share.ipynb`. Adds the six Krishna bat-11719 sessions to the joint `Bat_Transformer` fit and compares, per environment (**solo / fs / social / return-to-solo**), the pairwise **(residual) readout-weight cosine similarity** against the pairwise **Pearson correlation of 2-D rate maps**.

The same curated units are shared *within a day* (concat sort), so day 250812 (`solo_morning -> fs -> social_evening -> solo_evening`) tracks the same neurons across all four environments; 250813 adds solo / return-to-solo replicates. `fs` is the field station (same room as Dataset_2).

In [ ]:
# for auto-reloading external modules
%load_ext autoreload
%autoreload 2

import numpy as np
from matplotlib import pyplot as plt
from matplotlib.collections import LineCollection
from mpl_toolkits.mplot3d.art3d import Line3DCollection

%matplotlib inline

from scipy.io import loadmat
import glob
import h5py
from scipy.stats import pearsonr
import torch
from torch import nn
from sklearn.metrics import roc_curve, precision_recall_curve, auc
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import binned_statistic, wilcoxon
import pandas as pd
import pickle
import re
from datetime import datetime
import seaborn as sns
import ipympl

import os
import sys
sys.path.append(os.path.expanduser('~/Documents/code/utils'))
from plotting import plot_stars

# sys.path.append('/mnt/scratch09/yongrong/projects/bat-intelligence/src/digitalbat')
sys.path.append(os.path.expanduser('~/Documents/code/digitalbat')) 
from digitalbat.datasets.data import bat_data_loader
from digitalbat.models.former import Bat_Transformer
from digitalbat.models.glm import Bat_GLM
from digitalbat.models.layers import ELU1, TransformerEncoderRoPE
from digitalbat.training.trainer import bat_former_trainer
from digitalbat.utility.utils import auprc_per_unit, corr, interpolate_signal_with_nans, lowpass_filter, \
                                    corr, poisson_fraction_deviance_explained, poisson_fde_population, generate_2d_rate_map
from digitalbat.utility.utils import set_random_state, plot_2d_rate_map, plot_2d_trajectory_response, plot_2d_trajectory_response_multiple, \
                                    plot_3d_trajectory, resample_response, nearest_neighbor_score, OrthogonalityPermutationTest
from digitalbat.datasets.data import Data_Prep_for_Raw, bat_data_loader
from digitalbat.utility.interpreter import Saliency, PermutationImportance, ActivationMaximization, ShapAnalysis, Gradient_Bat_Transformer # IntegGradi
from digitalbat.utility.TrajectoryRF import Generate_Synthetic_Trajecories, Time_Resolved_TRF, Time_Resolved_Phase, Trajectory_Level_TRF

from digitalbat.utility.utils import (
        bat_palette, parse_bat, wilcoxon_safe, stars_for_p,
        add_significance_bracket, annotate_pvalue_corner,
    )

BAT_PALETTE = bat_palette()

sys.path.append(os.path.expanduser('~/Documents/code/digitalbat/notebooks'))
from notebook_helpers import (
    DATASET_TARGETS, DIST_THRESH, REW_LOCS, PERCH_LOCS,
    loc_bin_width_rm, N_BINS_PF,
    get_target_locs, compute_target_proximity,
)


def _attach_session_meta(df, session_col='session'):
    """Add `bat` and `date` columns parsed from session names like
    `Dataset_3_14543_240419` (bat = '14543', date = 2024-04-19)."""
    sess = df[session_col]
    df = df.copy()
    df['bat'] = sess.str.split('_').str[2]
    df['date'] = sess.str[-6:].apply(lambda s: datetime.strptime(s, '%y%m%d').date())
    return df

In [ ]:
seed = 1
set_random_state(seed)

device = 'cuda'

# Toggle synthetic-train comparison.  When True, the training/eval cells
# below run twice (real + synthetic Poisson-resampled spikes from a smoothed
# 3D rate map) and the real-vs-synthetic comparison cells fire.  Set False
# to skip all synthetic work.
RUN_SYNTHETIC = False

# Toggle trajectory-specific synthetic baseline: per-flight-cluster rate
# maps (clusters with > SYNTHETIC_TRAJ_MIN_CLUSTER_SIZE training flights
# get their own rate map; all others are pooled into one).  Independent
# of RUN_SYNTHETIC -- either or both may be enabled.
RUN_SYNTHETIC_TRAJ = False
SYNTHETIC_TRAJ_MIN_CLUSTER_SIZE = 3


In [ ]:
import re

# The six Krishna bat-11719 sessions that have ciholas behaviour
# (250813_fs is excluded upstream -- no behaviour).
KRISHNA_SESSIONS = [
    'Dataset_Krishna_11719_250812_solo_morning',
    'Dataset_Krishna_11719_250812_fs',
    'Dataset_Krishna_11719_250812_social_evening',
    'Dataset_Krishna_11719_250812_solo_evening',
    'Dataset_Krishna_11719_250813_solo_morning',
    'Dataset_Krishna_11719_250813_solo_evening',
]

# Environment label per Krishna session.  On 250812 the recording order is
# solo_morning -> fs (field station, same room as Dataset_2) -> social_evening
# -> solo_evening, so solo_evening is the post-social 'return to solo'.  250813
# has no fs/social; we keep the same suffix-based labels for consistency.
def session_env(session_name):
    if '_fs' in session_name:
        return 'fs'
    if 'social' in session_name:
        return 'social'
    if 'solo_evening' in session_name:
        return 'return to solo'
    if 'solo_morning' in session_name:
        return 'solo'
    return 'lab'  # the non-Krishna default sessions

ENV_ORDER = ['solo', 'fs', 'social', 'return to solo']
ENV_PALETTE = dict(zip(ENV_ORDER, sns.color_palette('colorblind', len(ENV_ORDER))))

# Krishna session names (Dataset_Krishna_11719_250812_solo_morning) put the
# 6-digit YYMMDD as the 4th token, not the last one, so the default
# `_attach_session_meta` (str[-6:]) breaks.  Override with a robust parser.
_DATE_RE = re.compile(r'_(\d{6})(?:_|$)')

def _attach_session_meta(df, session_col='session'):
    df = df.copy()
    sess = df[session_col].astype(str)
    df['bat'] = sess.str.split('_').str[2]
    def _date(s):
        m = _DATE_RE.search(s) or re.search(r'(\d{6})$', s)
        return datetime.strptime(m.group(1), '%y%m%d').date()
    df['date'] = sess.apply(_date)
    df['env'] = sess.apply(session_env)
    return df

# Keep bat-colored plots from KeyError-ing on the new bat.
if '11719' not in BAT_PALETTE:
    BAT_PALETTE['11719'] = sns.color_palette('tab10', 10)[len(BAT_PALETTE) % 10]


## Dataloader

In [ ]:
from digitalbat.datasets.defaults import DEFAULT_SESSION_NAMES, make_data_config

# Joint fit: the 17 canonical sessions + the 6 Krishna bat-11719 sessions.
session_names = list(DEFAULT_SESSION_NAMES) + KRISHNA_SESSIONS
data_config = make_data_config(session_names=session_names)
dataloaders, behavior_dim, unit_num_dict, meta_info_dict = bat_data_loader(**data_config)

print(f'{len(session_names)} sessions, behavior_dim={behavior_dim}')
print('Krishna unit counts:', {s.replace("Dataset_Krishna_11719_", ""): unit_num_dict[s]
                                for s in KRISHNA_SESSIONS})
dataloaders, behavior_dim, unit_num_dict


In [ ]:
# --- Flight cluster type distribution for the Krishna sessions (all splits + test) ---
# With split_each_trajectory_flag, each dataset item is one flight, so these are
# counts of flights per cluster type.
ks = sorted(KRISHNA_SESSIONS, key=lambda s: (s.split('_')[3], ENV_ORDER.index(session_env(s))))
ncols = 3
nrows = int(np.ceil(len(ks) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.3, nrows * 2.5), squeeze=False)
for k, s in enumerate(ks):
    ax = axes[k // ncols, k % ncols]
    ft_all = np.concatenate([dataloaders[sp][s].dataset.flight_type_index.detach().cpu().numpy()
                             for sp in ['train', 'valid', 'test']])
    ft_test = dataloaders['test'][s].dataset.flight_type_index.detach().cpu().numpy()
    uv, c = np.unique(ft_all, return_counts=True)
    uvt, ct = np.unique(ft_test, return_counts=True)
    ax.bar(uv, c, color='gray', label='all splits')
    ax.bar(uvt, ct, color='k', label='test')
    ax.set_xlabel('flight cluster type'); ax.set_ylabel('# flights')
    ax.set_title(f'{session_env(s)} {s.split("_")[3]}  (n={len(ft_all)} flights)', fontsize=9)
for k in range(len(ks), nrows * ncols):
    axes[k // ncols, k % ncols].set_visible(False)
axes[0, 0].legend(loc='best', frameon=False, fontsize=8)
fig.suptitle('Flight cluster type distribution (bat 11719)', y=1.02)
plt.tight_layout(); plt.show()


## Transformer model

In [ ]:
from digitalbat.models.former_defaults import make_model_config

model_config = make_model_config(behavior_dim, unit_num_dict, meta_info_dict)
bat_former = Bat_Transformer(**model_config)
print(f'{sum(p.numel() for p in bat_former.parameters()):,} parameters; '
      f'{len(unit_num_dict)} readout heads')


## Training

Fits a fresh joint model (defaults + 6 Krishna sessions) under a new `DATA_TAG`. Skips training if the checkpoint already exists.

In [ ]:
# Trainer hyperparameters; model_config is set in the previous cell.
from digitalbat.models.former_defaults import DEFAULT_LR, DEFAULT_WEIGHT_DECAY

os.makedirs('./out/bat_former', exist_ok=True)
lr_transformer = DEFAULT_LR
weight_decay_transformer = DEFAULT_WEIGHT_DECAY


In [ ]:
from digitalbat.models.former_defaults import make_model_name, make_trainer_config

# New data tag (defaults + 6 Krishna sessions) so this fits a FRESH joint model
# with Krishna readout heads, rather than loading the 17-session '_acc50'
# checkpoint shared with model_comparison.ipynb (which lacks the Krishna heads).
DATA_TAG = '_acc50_krishna6'

set_random_state(seed)
bat_former = Bat_Transformer(**model_config)
model_name = make_model_name(
    seed, lr_transformer, weight_decay_transformer,
    out_dir='./out/bat_former', suffix=DATA_TAG)
trainer_config = make_trainer_config(
    model=bat_former, dataloaders=dataloaders, device=device,
    model_name=model_name, lr=lr_transformer, weight_decay=weight_decay_transformer,
)
if os.path.exists(model_name):
    print(f'checkpoint exists ({model_name}) -- will load below.')
else:
    print(f'training {model_name} ...')
    best_acc, _, acc_vals = bat_former_trainer(**trainer_config)
    print(f'best val score: {best_acc:.4f}')


## Evaluation

In [ ]:
def evaluate_model(model, dataloaders, valid_fn, device, flight_filter=None, meta_info_dict=None):
    """
    Evaluate model on test data.
    
    Args:
        model: trained model in eval mode
        dataloaders: dict with 'test' key
        valid_fn: function(y_true, y_pred) -> scores array
        device: torch device
        flight_filter: None (all), 'ood', or 'indomain'
        meta_info_dict: required if flight_filter is not None
    
    Returns:
        dict mapping session_name -> scores array
    """
    results = {}
    model.eval()
    
    for session_name, session_data in dataloaders['test'].items():
        # Get flight type filter if needed
        if flight_filter == 'ood':
            target_types = np.array(meta_info_dict['flight_type_test_ood'][session_name], dtype=int)
            if target_types.size == 0:
                results[session_name] = np.array([])
                continue
        elif flight_filter == 'indomain':
            target_types = np.array(meta_info_dict['flight_type_test_indomain'][session_name], dtype=int)
        else:
            target_types = None
        
        y_val_list, y_pred_list = [], []
        for batch in session_data:
            X_val = batch.behavior.to(device)
            y_val = batch.spike.to(device)
            mask_val = batch.mask.to(device)
            
            # Filter by flight type if needed
            if target_types is not None:
                batch_mask = np.isin(batch.flight_type_index.numpy(), target_types)
                if not batch_mask.any():
                    continue
                X_val = X_val[batch_mask]
                y_val = y_val[batch_mask]
                mask_val = mask_val[batch_mask]
            
            y_pred = model(session_name, X_val, mask_val)
            y_val_list.append(y_val[mask_val == 1.0].detach().cpu().numpy())
            y_pred_list.append(y_pred[mask_val == 1.0].detach().cpu().numpy())
        
        if y_val_list:
            y_val_np = np.vstack(y_val_list)
            y_pred_np = np.vstack(y_pred_list)
            results[session_name] = valid_fn(y_val_np, y_pred_np)
        else:
            results[session_name] = np.array([])
    
    return results

In [ ]:
# Load the trained encoder and evaluate on the (real) test set.
metrics = {
    'corr': lambda y_true, y_pred: corr(y_true, y_pred, axis=0),
    'fde': lambda y_true, y_pred: poisson_fde_population(y_true, y_pred),
}
flight_filters = [None, 'ood', 'indomain']  # None = all flights

bat_former.load_state_dict(torch.load(model_name, map_location=device))
bat_former = bat_former.to(device)
bat_former.eval()

transformer_results = {}
for metric_name, valid_fn in metrics.items():
    for flight_filter in flight_filters:
        key = f"{metric_name}_{flight_filter or 'all'}"
        transformer_results[key] = evaluate_model(
            bat_former, dataloaders, valid_fn, device,
            flight_filter=flight_filter, meta_info_dict=meta_info_dict,
        )
        all_scores = np.hstack([v for v in transformer_results[key].values() if len(v) > 0])
        stat = np.nanmedian if 'fde' in key else np.nanmean
        print(f'{key}: {stat(all_scores):.4f}')

# Krishna-only summary
print('\nKrishna corr_all (mean per session):')
for s in KRISHNA_SESSIONS:
    v = transformer_results['corr_all'][s]
    print(f'  {session_env(s):>15} {s.split("_")[3]}: {np.nanmean(v):.3f}  (n={len(v)})')


In [ ]:
# --- Histograms of per-neuron test corr / FDE, by Krishna environment vs other bats ---
rows = []
for s in dataloaders['test'].keys():
    rows.append(pd.DataFrame({
        'corr': np.asarray(transformer_results['corr_all'][s]),
        'fde': np.asarray(transformer_results['fde_all'][s]),
        'session': s,
        'group': session_env(s) if s in KRISHNA_SESSIONS else 'other bats',
    }))
score_df = pd.concat(rows, ignore_index=True)
present = [g for g in (ENV_ORDER + ['other bats']) if g in set(score_df['group'])]

specs = [('corr', 'Pearson corr', np.linspace(-0.5, 1.0, 40)),
         ('fde', 'Poisson FDE', np.linspace(-1.0, 1.0, 40))]  # FDE unbounded below -> bound the AXIS, not the data
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (col, label, bins) in zip(axes, specs):
    sns.histplot(data=score_df, x=col, hue='group', hue_order=present, bins=bins,
                 stat='density', common_norm=False, element='step', fill=False, ax=ax)
    ax.set_xlim(bins[0], bins[-1])
    ax.axvline(0, color='k', lw=0.5, ls='--')
    ax.set_xlabel(f'{label} (per-neuron, test)')
axes[0].set_title('Test correlation')
axes[1].set_title('Test FDE')
plt.tight_layout(); plt.show()

print('Median per-neuron test scores by group:')
print(score_df.groupby('group')[['corr', 'fde']].median().reindex(present).to_string())


## Single-cell rate maps across environments (day 250812)

Two 39×4 figures (**row = cell, col = environment**) for the same 39 concat-sorted units on day 250812. The first shows the **binned (smoothed) rate map**, the second the **trajectory rate map** (flight-path samples colored by spike count). Each row shares one color scale across its four environments so a cell’s tuning can be compared solo → fs → social → return. These use observed spikes only (independent of the per-neuron table below).

In [ ]:
# --- Cache per-timepoint flight position + observed spikes for the four
# day-250812 sessions (same 39-unit concat sort: row i == same unit across
# environments) for the 39x4 single-cell rate-map / trajectory figures. ---
DAY = '250812'
viz_sessions = sorted([s for s in KRISHNA_SESSIONS if f'_{DAY}_' in s],
                      key=lambda s: ENV_ORDER.index(session_env(s)))
viz_envs = [session_env(s) for s in viz_sessions]
dt_rm = data_config['bin_width']

flight_xy, flight_spk, flight_lims = {}, {}, {}
for s in viz_sessions:
    Xl, yl, ml = [], [], []
    for b in dataloaders['test'][s]:
        Xl.append(b.behavior.numpy()); yl.append(b.spike.numpy()); ml.append(b.mask.numpy())
    Xa, ya, ma = np.vstack(Xl), np.vstack(yl), np.vstack(ml)
    bm = meta_info_dict['behavior_mean'][s]; bs = meta_info_dict['behavior_std'][s]
    lx = Xa[:, :, 0][ma == 1.0] * bs[0, 0, 0] + bm[0, 0, 0]
    ly = Xa[:, :, 1][ma == 1.0] * bs[0, 0, 1] + bm[0, 0, 1]
    flight_xy[s] = (lx, ly)
    flight_spk[s] = ya[ma == 1.0]                       # (n_valid_timepoints, n_neurons)
    flight_lims[s] = (meta_info_dict['r_lim'][s][0], meta_info_dict['r_lim'][s][1])

n_cells_viz = flight_spk[viz_sessions[0]].shape[1]
assert all(flight_spk[s].shape[1] == n_cells_viz for s in viz_sessions), 'unit count mismatch'
print(f'{n_cells_viz} cells x {len(viz_sessions)} environments {viz_envs}; '
      f'valid flight bins per env: {[flight_spk[s].shape[0] for s in viz_sessions]}')


In [ ]:
# --- Figure 1: binned (smoothed) rate maps, row = cell, col = environment ---
# Observed spikes only.  Each row (cell) shares one color scale across its four
# environments so brightness is directly comparable; peak rate is printed at left.
sm_maps = {s: [] for s in viz_sessions}; sm_edges = {}
for s in viz_sessions:
    lx, ly = flight_xy[s]; spk = flight_spk[s]
    xlim, ylim = flight_lims[s]
    for i in range(n_cells_viz):
        xe, ye, _, sm = generate_2d_rate_map(lx, ly, spk[:, i], xlim, ylim, loc_bin_width_rm, dt_rm)
        sm_maps[s].append(sm)
    sm_edges[s] = (xe, ye)

nrow, ncol = n_cells_viz, len(viz_sessions)
fig, axes = plt.subplots(nrow, ncol, figsize=(ncol * 1.7, nrow * 1.15), squeeze=False)
for i in range(nrow):
    row_vmax = max(np.nanmax(sm_maps[s][i]) for s in viz_sessions)
    row_vmax = row_vmax if row_vmax > 0 else 1.0
    for j, s in enumerate(viz_sessions):
        xe, ye = sm_edges[s]; ax = axes[i, j]
        ax.imshow(sm_maps[s][i].T, origin='lower', vmin=0, vmax=row_vmax,
                  extent=[xe[0], xe[-1], ye[0], ye[-1]], aspect='auto', cmap='hot_r')
        ax.set_xticks([]); ax.set_yticks([])
        if i == 0:
            ax.set_title(viz_envs[j], fontsize=10)
    pt = meta_info_dict['pType'][viz_sessions[0]][i]   # '' for filtered (empty-array) units
    pt = pt if isinstance(pt, str) else ''
    axes[i, 0].set_ylabel(f'cell {i}\n{row_vmax:.1f} Hz\n{pt}', fontsize=6,
                          rotation=0, ha='right', va='center')
fig.suptitle(f'Binned rate maps (observed) — bat 11719, day {DAY}  '
             '(row = cell, col = environment; per-row shared scale)', y=1.002, fontsize=12)
plt.tight_layout(); plt.show()


In [ ]:
# --- Figure 2: trajectory rate maps, row = cell, col = environment ---
# Flight-path samples scatter-colored by observed spike count (per 0.1 s bin),
# high counts drawn on top.  Per-row shared color scale across environments.
fig, axes = plt.subplots(nrow, ncol, figsize=(ncol * 1.7, nrow * 1.15), squeeze=False)
for i in range(nrow):
    row_vmax = max(int(flight_spk[s][:, i].max()) for s in viz_sessions)
    row_vmax = row_vmax if row_vmax > 0 else 1
    for j, s in enumerate(viz_sessions):
        lx, ly = flight_xy[s]; sc = flight_spk[s][:, i]
        xlim, ylim = flight_lims[s]; order = np.argsort(sc)   # high spike counts on top
        ax = axes[i, j]
        ax.scatter(lx[order], ly[order], c=sc[order], cmap='hot', s=1.2,
                   vmin=0, vmax=row_vmax, linewidths=0)
        ax.set_xlim(xlim); ax.set_ylim(ylim)
        ax.set_xticks([]); ax.set_yticks([]); ax.set_aspect('auto')
        if i == 0:
            ax.set_title(viz_envs[j], fontsize=10)
    axes[i, 0].set_ylabel(f'cell {i}\n{int(row_vmax)} spk', fontsize=6,
                          rotation=0, ha='right', va='center')
fig.suptitle(f'Trajectory rate maps (observed spike count) — bat 11719, day {DAY}  '
             '(row = cell, col = environment; per-row shared scale)', y=1.002, fontsize=12)
plt.tight_layout(); plt.show()


## Per-neuron table (readout weights, pType, mean FR)

In [ ]:
inds_sessions = {} # a dictionary of neuronal indices for each session
pType_sessions = {}  # pType arrays from meta_info_dict, filtered to selected units
readout_weight_sessions = [] # readout weights for selected units from all sessions
for session_target in list(dataloaders['test'].keys()):
    readout_weights = bat_former.readouts[session_target][0].weight.detach().cpu().numpy()
    n_neurons = readout_weights.shape[0]
    inds = np.arange(n_neurons)
    inds_sessions[session_target] = inds
    pType_sessions[session_target] = np.asarray(meta_info_dict['pType'][session_target])
    readout_weight_sessions.append(readout_weights)
readout_weight_sessions = np.vstack(readout_weight_sessions)
print(f'readout_weight_sessions.shape: {readout_weight_sessions.shape}')

inds_sessions_df = pd.concat(
    [
        pd.DataFrame({
            "session_neuron_idx": inds,
            "session_name": session
        })
        for session, inds in inds_sessions.items()
    ],
    ignore_index=True
)
inds_sessions_df["global_neuron_idx"] = range(len(inds_sessions_df))

# Attach pType (putative cell type)
inds_sessions_df['pType'] = np.concatenate(
    [pType_sessions[s] for s in inds_sessions.keys()]
)

# Attach probe identity
inds_sessions_df['probe_id'] = np.concatenate(
    [meta_info_dict['probe_id'][s] for s in inds_sessions.keys()]
)

# Attach neuron position (µm)
positions = np.vstack([meta_info_dict['neuron_position'][s] for s in inds_sessions.keys()])
inds_sessions_df['neuron_pos_lateral'] = positions[:, 0]  # spikePos_um col0 = x
inds_sessions_df['neuron_pos_depth'] = positions[:, 1]    # spikePos_um col1 = y/depth

# Compute mean firing rate per neuron from test data
mean_fr_sessions = {}
for session_name, session_data in dataloaders['test'].items():
    y_list, mask_list = [], []
    for batch in session_data:
        y_list.append(batch.spike.numpy())
        mask_list.append(batch.mask.numpy())
    y_all = np.vstack(y_list)
    mask_all = np.vstack(mask_list)
    spikes_flat = y_all[mask_all == 1.0]
    mean_fr_sessions[session_name] = spikes_flat.mean(axis=0) / data_config['bin_width']
inds_sessions_df['mean_fr'] = np.concatenate(
    [mean_fr_sessions[s] for s in inds_sessions.keys()]
)

# Attach per-neuron metrics from transformer_results. evaluate_model
# returns np.array([]) for sessions with no flights matching the filter
# (e.g., when meta_info_dict['flight_type_test_ood'][session] is empty),
# so pad those with NaNs of length n_neurons to keep length alignment
# with inds_sessions_df.
for key in transformer_results:
    chunks = []
    for s in inds_sessions.keys():
        scores = np.asarray(transformer_results[key][s])
        n = len(inds_sessions[s])
        chunks.append(scores if scores.size == n else np.full(n, np.nan))
    inds_sessions_df[key] = np.concatenate(chunks)

# # Attach per-neuron p-values from circular-shift null
# for key in transformer_pvals:
#     inds_sessions_df[f'p_{key}'] = np.concatenate(
#         [transformer_pvals[key][s] for s in inds_sessions.keys()]
#     )

# Store readout weights (unit-normed) directly in the dataframe
norms = np.linalg.norm(readout_weight_sessions, axis=1, keepdims=True)
readout_normed = readout_weight_sessions / norms
inds_sessions_df['readout_weight'] = list(readout_normed)
inds_sessions_df.loc[inds_sessions_df['pType'].apply(lambda x: isinstance(x, np.ndarray)), 'pType'] = 'filtered'
inds_sessions_df = _attach_session_meta(inds_sessions_df, session_col='session_name')
ct_df = inds_sessions_df[inds_sessions_df['pType'].isin(['pPYR', 'pINT'])].copy()

## Rate maps (observed spikes)

In [ ]:
# Compute 2D rate maps for all neurons in each session
# Uses observed spike counts (not model predictions)
dt_rm = data_config['bin_width']

session_rate_maps = {}  # session -> {'all': array(n_neurons, n_flat_bins), 'top_cluster': ...}
session_top_cluster = {}

for session_name, session_data in dataloaders['test'].items():
    X_list, y_list, mask_list, ftype_list = [], [], [], []
    for batch in session_data:
        X_list.append(batch.behavior.numpy())
        y_list.append(batch.spike.numpy())
        mask_list.append(batch.mask.numpy())
        ftype_list.append(batch.flight_type_index.numpy())

    X_all = np.vstack(X_list)
    y_all = np.vstack(y_list)
    mask_all = np.vstack(mask_list)
    ftype_all = np.hstack(ftype_list)
    n_neurons = y_all.shape[2]

    # Denormalize x, y positions
    bm = meta_info_dict['behavior_mean'][session_name]
    bs = meta_info_dict['behavior_std'][session_name]
    loc_xx_lim = meta_info_dict['r_lim'][session_name][0, :]
    loc_yy_lim = meta_info_dict['r_lim'][session_name][1, :]

    # Consistent bin edges for this session
    x_edges = np.arange(loc_xx_lim[0], loc_xx_lim[1] + loc_bin_width_rm, loc_bin_width_rm)
    y_edges = np.arange(loc_yy_lim[0], loc_yy_lim[1] + loc_bin_width_rm, loc_bin_width_rm)
    n_flat = (len(x_edges) - 1) * (len(y_edges) - 1)

    # Most numerous flight cluster
    top_cluster = int(pd.Series(ftype_all).value_counts().index[0])
    session_top_cluster[session_name] = top_cluster

    rate_maps_dict = {}
    for label, flight_mask in [('all', np.ones(len(ftype_all), dtype=bool)),
                               ('top_cluster', ftype_all == top_cluster)]:
        X_sub = X_all[flight_mask]
        y_sub = y_all[flight_mask]
        mask_sub = mask_all[flight_mask]

        # Flatten valid timepoints
        loc_xx = (X_sub[:, :, 0][mask_sub == 1.0] * bs[0, 0, 0] + bm[0, 0, 0])
        loc_yy = (X_sub[:, :, 1][mask_sub == 1.0] * bs[0, 0, 1] + bm[0, 0, 1])
        spikes = y_sub[mask_sub == 1.0]  # (n_valid_timepoints, n_neurons)

        # Occupancy (shared across neurons)
        occupancy, _, _ = np.histogram2d(loc_xx, loc_yy, bins=[x_edges, y_edges])
        occupancy_time = occupancy * dt_rm

        # Rate map per neuron, NaN for unvisited bins
        maps = np.full((n_neurons, n_flat), np.nan)
        for neuron_idx in range(n_neurons):
            spike_map, _, _ = np.histogram2d(
                loc_xx, loc_yy, bins=[x_edges, y_edges],
                weights=spikes[:, neuron_idx],
            )
            with np.errstate(divide='ignore', invalid='ignore'):
                rmap = spike_map / occupancy_time  # NaN where unvisited
            maps[neuron_idx] = rmap.ravel()

        rate_maps_dict[label] = maps

    session_rate_maps[session_name] = rate_maps_dict
    print(f'{session_name}: {n_neurons} neurons, top_cluster={top_cluster}, '
          f'n_bins={n_flat}, n_valid_all={mask_all.sum():.0f}')

## Per-environment: (residual) readout cosine similarity vs rate-map correlation

For each Krishna session, the pairwise neuron-by-neuron **residual-readout cosine similarity** is compared against the pairwise **rate-map Pearson correlation** via a Mantel test (Spearman $\rho$, 1000-permutation null).

In [ ]:
# --- Per-environment Mantel: (residual) readout cosine sim vs rate-map corr ---
from scipy.stats import rankdata

def _spearman(x, y):
    x = np.asarray(x, float); y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)   # drop NaN pairs (constant rate-map / FR rows)
    if m.sum() < 2:
        return np.nan
    rx, ry = rankdata(x[m]), rankdata(y[m])
    if rx.std() == 0 or ry.std() == 0:
        return np.nan
    return np.corrcoef(rx, ry)[0, 1]

def _mantel(a_sym, b_vec, rng, n_perms):
    """Mantel: permute labels of square matrix a_sym, Spearman-correlate its
    upper triangle with b_vec. Returns (r_obs, p, perm_rs)."""
    N = a_sym.shape[0]
    iu, ju = np.triu_indices(N, k=1)
    r_obs = _spearman(a_sym[iu, ju], b_vec)
    perm_rs = np.empty(n_perms)
    for pi in range(n_perms):
        perm = rng.permutation(N)
        perm_rs[pi] = _spearman(a_sym[np.ix_(perm, perm)][iu, ju], b_vec)
    p = (np.sum(perm_rs >= r_obs) + 1) / (n_perms + 1)
    return r_obs, p, perm_rs

def add_residual_readout(df):
    """Per-session: project the firing-rate axis out of the (unit-normed)
    readout weights, re-normalize.  Same recipe as the original notebook."""
    df = df.copy()
    df['readout_weight_residual'] = df['readout_weight']  # placeholder dtype
    for session_name, grp in df.groupby('session_name'):
        idx = grp.index
        W = np.stack(grp['readout_weight'].values)
        fr_c = grp['mean_fr'].values - grp['mean_fr'].values.mean()
        fr_axis = W.T @ fr_c
        nrm = np.linalg.norm(fr_axis)
        if nrm > 0:
            fr_axis = fr_axis / nrm
            Wr = W - np.outer(W @ fr_axis, fr_axis)
        else:
            Wr = W.copy()
        n = np.linalg.norm(Wr, axis=1, keepdims=True); n[n == 0] = 1.0
        Wr = Wr / n
        for j, ix in enumerate(idx):
            df.at[ix, 'readout_weight_residual'] = Wr[j]
    return df

# Well-fit neurons only (matches the original rate-map Mantel: corr_all > 0.2).
clu_df = inds_sessions_df.loc[inds_sessions_df['corr_all'] > 0.2].copy()
clu_df = add_residual_readout(clu_df)

n_mantel_perms = 1000
rng = np.random.default_rng(42)

mantel_records, pair_records = [], []
for readout_label, col in [('readout', 'readout_weight'),
                           ('residual readout', 'readout_weight_residual')]:
    for session_name in KRISHNA_SESSIONS:
        grp = clu_df[clu_df['session_name'] == session_name].reset_index(drop=True)
        if len(grp) < 3:
            continue
        neuron_idxs = grp['session_neuron_idx'].values
        maps = session_rate_maps[session_name]['all'][neuron_idxs]
        shared = ~np.any(np.isnan(maps), axis=0)            # bins valid for ALL neurons
        maps_clean = maps[:, shared]
        if maps_clean.shape[1] < 5:
            continue
        has_var = np.var(maps_clean, axis=1) > 0            # drop constant rate maps
        if has_var.sum() < 3:
            continue
        maps_v = maps_clean[has_var]
        grp_v = grp[has_var]
        N = len(grp_v)
        iu, ju = np.triu_indices(N, k=1)
        rate_corr = np.corrcoef(maps_v)
        W = np.stack(grp_v[col].values)
        cos_sim = W @ W.T
        r_obs, p, perm_rs = _mantel(rate_corr, cos_sim[iu, ju], rng, n_mantel_perms)
        mantel_records.append(dict(
            variant=readout_label, session=session_name,
            env=session_env(session_name), date=session_name.split('_')[3],
            r=r_obs, p=p, n_neurons=N, perm_rs=perm_rs))
        if readout_label == 'residual readout':
            pair_records.append(pd.DataFrame(dict(
                session=session_name, env=session_env(session_name),
                date=session_name.split('_')[3],
                cos_sim=cos_sim[iu, ju], rate_corr=rate_corr[iu, ju])))

mantel_df = _attach_session_meta(pd.DataFrame(mantel_records))
pair_df = pd.concat(pair_records, ignore_index=True)
mantel_df['stars'] = mantel_df['p'].apply(stars_for_p)
print(mantel_df.drop(columns='perm_rs').to_string(index=False))


In [ ]:
# Scatter: pairwise residual-readout cosine sim vs rate-map correlation, by env.
g = sns.lmplot(
    data=pair_df, x='rate_corr', y='cos_sim', col='env', col_order=ENV_ORDER,
    hue='date', scatter_kws=dict(s=6, alpha=0.25), line_kws=dict(color='k'),
    height=3.5, facet_kws=dict(sharex=True, sharey=True),
)
for env, ax in g.axes_dict.items():
    sub = mantel_df[(mantel_df['variant'] == 'residual readout') & (mantel_df['env'] == env)]
    txt = '\n'.join(f"{r['date']}: $\\rho$={r['r']:.2f}{r['stars']}" for _, r in sub.iterrows())
    ax.text(0.04, 0.96, txt, transform=ax.transAxes, va='top', ha='left', fontsize=8)
    ax.axhline(0, color='0.7', lw=0.5); ax.axvline(0, color='0.7', lw=0.5)
g.set_axis_labels('rate-map Pearson corr', 'residual readout cosine sim')
g.set_titles('{col_name}')
g.figure.suptitle('Pairwise residual-readout cosine vs rate-map correlation, by environment',
                  y=1.04)
plt.show()


In [ ]:
# Per-environment Mantel rho (readout & residual readout), with permutation null.
fig, ax = plt.subplots(figsize=(7.5, 4.5))
order, hue_order = ENV_ORDER, ['readout', 'residual readout']
sns.barplot(data=mantel_df, x='env', y='r', order=order, hue='variant',
            hue_order=hue_order, errorbar=None, alpha=0.45, ax=ax)
sns.stripplot(data=mantel_df, x='env', y='r', order=order, hue='variant',
              hue_order=hue_order, dodge=True, ax=ax, edgecolor='k',
              linewidth=0.5, size=9, legend=False)
# Significance stars above each point (offset by variant within env).
offsets = {'readout': -0.2, 'residual readout': 0.2}
for _, row in mantel_df.iterrows():
    x = order.index(row['env']) + offsets[row['variant']]
    if row['stars']:
        ax.text(x, row['r'] + 0.02, row['stars'], ha='center', va='bottom', fontsize=9)
ax.axhline(0, color='k', lw=0.5, ls='--')
ax.set_ylabel('Mantel $\\rho$ (cosine sim vs rate-map corr)')
ax.set_xlabel('environment')
ax.set_title('Readout / rate-map alignment per environment (bat 11719)')
ax.legend(title='', loc='upper right', fontsize=9)
plt.tight_layout(); plt.show()


### Same neurons across environments (day 250812)

Day 250812 shares one concat sort across all four environments, so we can ask how the readout cosine structure and the rate-map correlation structure of the **same neurons** reorganize across solo / fs / social / return.

In [ ]:
# Same 39 neurons across all four environments on day 250812 (concat sort).
# How stable is the (residual) readout cosine structure -- and the rate-map
# correlation structure -- as the bat moves solo -> fs -> social -> return?
DAY = '250812'
day_sessions = sorted([s for s in KRISHNA_SESSIONS if f'_{DAY}_' in s],
                      key=lambda s: ENV_ORDER.index(session_env(s)))
envs = [session_env(s) for s in day_sessions]

# Common neuron set: corr_all > 0.2 in ALL four environments.
good = None
for s in day_sessions:
    idx = set(inds_sessions_df.loc[(inds_sessions_df['session_name'] == s) &
                                   (inds_sessions_df['corr_all'] > 0.2), 'session_neuron_idx'])
    good = idx if good is None else (good & idx)
common = np.array(sorted(good))
print(f'{len(common)} neurons pass corr_all>0.2 in all four {DAY} environments')

read_mats, rate_mats = {}, {}
for s in day_sessions:
    e = session_env(s)
    sub = (inds_sessions_df[inds_sessions_df['session_name'] == s]
           .set_index('session_neuron_idx').loc[common])
    W = np.stack(sub['readout_weight'].values)                 # already unit-normed
    fr_c = sub['mean_fr'].values - sub['mean_fr'].values.mean()
    fr_axis = W.T @ fr_c; nrm = np.linalg.norm(fr_axis)
    Wr = W - np.outer(W @ fr_axis, fr_axis) if nrm > 0 else W.copy()
    n = np.linalg.norm(Wr, axis=1, keepdims=True); n[n == 0] = 1.0
    read_mats[e] = (Wr / n) @ (Wr / n).T
    maps = session_rate_maps[s]['all'][common]
    shared = ~np.any(np.isnan(maps), axis=0)
    rate_mats[e] = np.corrcoef(maps[:, shared])

iu, ju = np.triu_indices(len(common), k=1)
def _cross(mats):
    M = np.zeros((len(envs), len(envs)))
    for a in range(len(envs)):
        for b in range(len(envs)):
            M[a, b] = _spearman(mats[envs[a]][iu, ju], mats[envs[b]][iu, ju])
    return M
read_cross, rate_cross = _cross(read_mats), _cross(rate_mats)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.6))
for ax, (M, ttl) in zip(axes, [(read_cross, 'residual readout cosine structure'),
                               (rate_cross, 'rate-map corr structure')]):
    sns.heatmap(M, annot=True, fmt='.2f', vmin=-1, vmax=1, cmap='RdBu_r',
                xticklabels=envs, yticklabels=envs, square=True, ax=ax,
                cbar_kws=dict(label='Spearman $\\rho$'))
    ax.set_title(f'{ttl}\ncross-environment stability (same {len(common)} neurons)')
plt.setp(axes[0].get_xticklabels(), rotation=30, ha='right')
plt.setp(axes[1].get_xticklabels(), rotation=30, ha='right')
plt.tight_layout(); plt.show()

# Within-environment readout-vs-ratemap alignment on the SAME common neuron set.
print('\nWithin-environment Mantel (readout structure vs rate-map structure),'
      f' same {len(common)} neurons:')
for e in envs:
    print(f'  {e:>15}: rho = {_spearman(read_mats[e][iu, ju], rate_mats[e][iu, ju]):.3f}')


## Non-flight FR correlation across environments

The pairwise **non-flight (rest) FR correlation** — Pearson correlation of sqrt-transformed non-flight spike counts — is computed per environment and (i) related to the flight readout / rate-map structure via per-environment Mantel tests, and (ii) compared across environments for the same day-250812 neurons (third panel alongside readout and rate-map structure).

In [ ]:
# --- Load non-flight spike data for the Krishna sessions ---
# only_flight_flag=True gives flight_ind; only_flight_flag=False gives every bin;
# the complement is the non-flight bins (rest / perch / reward periods between
# flights).  Mirrors the main notebook's non-flight loader.
nonflight_spikes = {}      # session -> (n_neurons, n_nonflight_bins)
for session_name in KRISHNA_SESSIONS:
    session_folder = data_config['parent_folder'] + session_name
    su_file_path = (data_config['parent_folder']
                    + '/To_transfer/out_Gunits_probe*_' + session_name + '_with_metrics.mat')
    dp_common = dict(
        parent_folder=session_folder, su_file_path=su_file_path,
        use_speed_head_dir=False, use_IMU_flag=False, use_click_flag=False,
        use_LFP_flag=False, use_ripple_flag=False,
        spike_count_flag=data_config['spike_count_flag'], bin_width=data_config['bin_width'],
    )
    dp_flight = Data_Prep_for_Raw(only_flight_flag=True, **dp_common)
    dp_flight.get_all_behavior(); dp_flight.get_all_spike()
    flight_ind = dp_flight.flight_ind

    dp_all = Data_Prep_for_Raw(only_flight_flag=False, **dp_common)
    dp_all.get_all_behavior(); all_spike = dp_all.get_all_spike()
    nonflight_ind = np.setdiff1d(np.arange(all_spike.shape[1]), flight_ind)
    nonflight_spikes[session_name] = all_spike[:, nonflight_ind]
    print(f'{session_env(session_name):>15} {session_name.split("_")[3]}: '
          f'{all_spike.shape[1]} total, {len(flight_ind)} flight, '
          f'{len(nonflight_ind)} non-flight bins, {all_spike.shape[0]} neurons')


In [ ]:
# --- Per-environment Mantel: non-flight FR correlation vs (residual) readout & rate map ---
# Non-flight FR correlation = Pearson corr of sqrt-transformed non-flight spike
# counts (variance-stabilized), as in the main notebook.  Per Krishna session we
# ask whether that rest correlation structure tracks the flight readout / rate-map
# structure, and how this differs across environments.
nf_mantel_records = []
for session_name in KRISHNA_SESSIONS:
    grp = clu_df[clu_df['session_name'] == session_name].reset_index(drop=True)
    if len(grp) < 3:
        continue
    neuron_idxs = grp['session_neuron_idx'].values
    spk = nonflight_spikes[session_name][neuron_idxs]
    has_var = np.var(spk, axis=1) > 0
    if has_var.sum() < 3:
        continue
    spk_v = spk[has_var]; grp_v = grp[has_var]
    N = len(grp_v); iu, ju = np.triu_indices(N, k=1)
    nf_corr = np.corrcoef(np.sqrt(spk_v))

    Wr = np.stack(grp_v['readout_weight_residual'].values); cos_resid = Wr @ Wr.T
    W = np.stack(grp_v['readout_weight'].values); cos_read = W @ W.T
    for label, ref in [('residual readout', cos_resid), ('readout', cos_read)]:
        r, p, _ = _mantel(nf_corr, ref[iu, ju], rng, n_mantel_perms)
        nf_mantel_records.append(dict(reference=label, session=session_name,
            env=session_env(session_name), date=session_name.split('_')[3],
            r=r, p=p, n_neurons=N))

    # rate-map corr on the SAME non-flight neuron subset (shared valid, non-constant bins)
    maps = session_rate_maps[session_name]['all'][neuron_idxs][has_var]
    shared = ~np.any(np.isnan(maps), axis=0); maps_c = maps[:, shared]
    if maps_c.shape[1] >= 5:
        both = np.var(maps_c, axis=1) > 0
        if both.sum() >= 3:
            ns = int(both.sum()); ius, jus = np.triu_indices(ns, k=1)
            rate_corr = np.corrcoef(maps_c[both]); nf_sub = np.corrcoef(np.sqrt(spk_v[both]))
            r, p, _ = _mantel(nf_sub, rate_corr[ius, jus], rng, n_mantel_perms)
            nf_mantel_records.append(dict(reference='rate map', session=session_name,
                env=session_env(session_name), date=session_name.split('_')[3],
                r=r, p=p, n_neurons=ns))

nf_mantel_df = _attach_session_meta(pd.DataFrame(nf_mantel_records))
nf_mantel_df['stars'] = nf_mantel_df['p'].apply(stars_for_p)
print(nf_mantel_df.to_string(index=False))

ref_order = ['residual readout', 'readout', 'rate map']
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(data=nf_mantel_df, x='env', y='r', order=ENV_ORDER, hue='reference',
            hue_order=ref_order, errorbar=None, alpha=0.45, ax=ax)
sns.stripplot(data=nf_mantel_df, x='env', y='r', order=ENV_ORDER, hue='reference',
              hue_order=ref_order, dodge=True, ax=ax, edgecolor='k', linewidth=0.5,
              size=8, legend=False)
ax.axhline(0, color='k', lw=0.5, ls='--')
ax.set_ylabel('Mantel $\\rho$ (non-flight FR corr vs reference)')
ax.set_xlabel('environment')
ax.set_title('Non-flight FR correlation vs flight readout / rate-map structure (bat 11719)')
ax.legend(title='', loc='upper right', fontsize=8)
plt.tight_layout(); plt.show()


In [ ]:
# --- Cross-environment stability of the non-flight FR correlation structure ---
# Same neurons across the four day-250812 environments (reuses common / envs /
# read_mats / rate_mats / read_cross / rate_cross / _cross / iu / ju from the
# readout/rate-map cross-env cell above).
nf_mats = {session_env(s): np.corrcoef(np.sqrt(nonflight_spikes[s][common])) for s in day_sessions}
nf_cross = _cross(nf_mats)

fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.6))
for ax, (M, ttl) in zip(axes, [(read_cross, 'residual readout cosine'),
                               (rate_cross, 'rate-map corr'),
                               (nf_cross, 'non-flight FR corr')]):
    sns.heatmap(M, annot=True, fmt='.2f', vmin=-1, vmax=1, cmap='RdBu_r',
                xticklabels=envs, yticklabels=envs, square=True, ax=ax,
                cbar_kws=dict(label='Spearman $\\rho$'))
    ax.set_title(f'{ttl}\ncross-environment stability (same {len(common)} neurons)')
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
plt.tight_layout(); plt.show()

# Within-environment: does the rest correlation track the flight structure?
print(f'Within-environment Mantel (non-flight FR corr vs ...), same {len(common)} neurons:')
for e in envs:
    r_rd = _spearman(nf_mats[e][iu, ju], read_mats[e][iu, ju])
    r_rm = _spearman(nf_mats[e][iu, ju], rate_mats[e][iu, ju])
    print(f'  {e:>15}: vs residual readout rho={r_rd:+.3f} | vs rate map rho={r_rm:+.3f}')


## Environment-keyed trajectory projection (`embed_traj` per room)

Model variant: a **separate `embed_traj` linear projection per environment** (field station vs indoor lab), enabled via the new `session_to_room` argument to `Bat_Transformer`. The same per-session-normalised position is projected differently in FS vs lab, so cross-environment remapping is carried by the trajectory encoder. Trained as a separate checkpoint and compared against the shared-projection baseline above (FDE / corr per environment).

In [ ]:
# --- Environment-keyed trajectory projection: a separate embed_traj per room ---
# Field station (outdoor; Krishna *_fs + Dataset_2, the same physical room) gets
# its own trajectory projection; all indoor lab sessions share one.  The same
# (per-session normalised) position is then projected differently in FS vs lab,
# so cross-environment remapping is carried by the trajectory encoder rather than
# absorbed by the per-session readouts.  Trained as a SEPARATE checkpoint.
def session_room(s):
    # FS = Krishna field-station sessions + Dataset_2 (same room); else indoor lab.
    if s.endswith('_fs') or s.startswith('Dataset_2_'):
        return 'field_station'
    return 'lab'

session_to_room = {s: session_room(s) for s in session_names}
print('rooms:', pd.Series(session_to_room).value_counts().to_dict())
print('field_station sessions:', [s for s, r in session_to_room.items() if r == 'field_station'])

model_config_env = make_model_config(behavior_dim, unit_num_dict, meta_info_dict,
                                     session_to_room=session_to_room)
DATA_TAG_ENV = '_acc50_krishna6_envtraj'
set_random_state(seed)
bat_former_env = Bat_Transformer(**model_config_env)
model_name_env = make_model_name(seed, lr_transformer, weight_decay_transformer,
                                 out_dir='./out/bat_former', suffix=DATA_TAG_ENV)
trainer_config_env = make_trainer_config(
    model=bat_former_env, dataloaders=dataloaders, device=device,
    model_name=model_name_env, lr=lr_transformer, weight_decay=weight_decay_transformer,
)
if os.path.exists(model_name_env):
    print(f'checkpoint exists ({model_name_env}) -- will load below.')
else:
    print(f'training {model_name_env} ...')
    best_acc_env, _, acc_vals_env = bat_former_trainer(**trainer_config_env)
    print(f'best val score: {best_acc_env:.4f}')


In [ ]:
# --- Evaluate the env-keyed model and compare to the shared-projection baseline ---
bat_former_env.load_state_dict(torch.load(model_name_env, map_location=device))
bat_former_env = bat_former_env.to(device).eval()

transformer_results_env = {}
for metric_name, valid_fn in metrics.items():
    for flight_filter in flight_filters:
        key = f"{metric_name}_{flight_filter or 'all'}"
        transformer_results_env[key] = evaluate_model(
            bat_former_env, dataloaders, valid_fn, device,
            flight_filter=flight_filter, meta_info_dict=meta_info_dict)

# Paired per-neuron comparison on the Krishna sessions (baseline `transformer_results`
# is the shared-embed model from the Evaluation section above).
cmp_df = pd.concat([pd.DataFrame({
    'session': s, 'env': session_env(s), 'room': session_to_room[s],
    'fde_shared': transformer_results['fde_all'][s],
    'fde_envkey': transformer_results_env['fde_all'][s],
    'corr_shared': transformer_results['corr_all'][s],
    'corr_envkey': transformer_results_env['corr_all'][s],
}) for s in KRISHNA_SESSIONS], ignore_index=True)

summ = cmp_df.groupby('env')[['fde_shared', 'fde_envkey', 'corr_shared', 'corr_envkey']].median().reindex(ENV_ORDER)
print('Median per-neuron test scores (shared vs env-keyed embed_traj):')
print(summ.to_string())
print('\nPaired Wilcoxon (env-keyed > shared) per environment:')
for e in ENV_ORDER:
    g = cmp_df[cmp_df['env'] == e]
    try:
        _, p_fde = wilcoxon(g['fde_envkey'], g['fde_shared'], alternative='greater')
    except ValueError:
        p_fde = np.nan
    print(f'  {e:>15}: dFDE_median={ (g["fde_envkey"]-g["fde_shared"]).median():+.4f}  p={p_fde:.3g}  (n={len(g)})')

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, metric in zip(axes, ['fde', 'corr']):
    sub = cmp_df.melt(id_vars=['env'], value_vars=[f'{metric}_shared', f'{metric}_envkey'],
                      var_name='model', value_name=metric)
    sub['model'] = sub['model'].str.replace(f'{metric}_', '', regex=False)
    sns.boxplot(data=sub, x='env', y=metric, order=ENV_ORDER, hue='model',
                hue_order=['shared', 'envkey'], showfliers=False, ax=ax)
    if metric == 'fde':
        ax.set_ylim(-1, 1)  # FDE unbounded below -> bound the AXIS, not the data
    ax.axhline(0, color='k', lw=0.5, ls='--')
    ax.set_xlabel('environment'); ax.set_ylabel(metric.upper())
    ax.set_title(f'{metric.upper()}: shared vs env-keyed embed_traj')
    ax.legend(title='', fontsize=8)
plt.tight_layout(); plt.show()
